In [1]:
RANDOM_STATE = 21

## 7. Сравнение LightGBM, CatBoost и XGBoost

In [2]:
%%time
# Задание 7. Загрузка выборок и выделение целевой переменной

from pathlib import Path

import pandas as pd

DATA_DIR = Path("data")
TARGET = "IsBadBuy"

df_train = pd.read_csv(DATA_DIR / "df_train.csv")
df_valid = pd.read_csv(DATA_DIR / "df_valid.csv")

X_train = df_train.drop(columns=[TARGET])
y_train = df_train[TARGET]

X_valid = df_valid.drop(columns=[TARGET])
y_valid = df_valid[TARGET]

assert X_train.columns.equals(X_valid.columns)

print(f"Train: {X_train.shape}")
print(f"Validation: {X_valid.shape}")

Train: (24391, 33)
Validation: (24255, 33)
CPU times: total: 656 ms
Wall time: 1.33 s


In [3]:
%%time
# Задание 7. Подготовка общих признаков для сравнения библиотек

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder

categorical_columns = [
    "Auction", "Make", "Model", "Trim", "SubModel", "Color",
    "Transmission", "WheelType", "Nationality", "Size",
    "TopThreeAmericanName", "BYRNO", "VNZIP1", "VNST",
]

excluded_columns = {
    "RefId", "PurchDate", "WheelTypeID",
    "PRIMEUNIT", "AUCGUART", "IsOnlineSale",
}

numeric_columns = [
    column for column in X_train.columns
    if column not in categorical_columns and column not in excluded_columns
]

preprocessor = ColumnTransformer(
    [
        (
            "categorical",
            make_pipeline(
                SimpleImputer(strategy="constant", fill_value="__MISSING__"),
                OneHotEncoder(handle_unknown="ignore"),
            ),
            categorical_columns,
        ),
        ("numeric", SimpleImputer(strategy="median"), numeric_columns),
    ],
    sparse_threshold=1.0,
)

X_train_encoded = preprocessor.fit_transform(X_train)
X_valid_encoded = preprocessor.transform(X_valid)

assert X_train_encoded.shape[0] == len(y_train)
assert X_valid_encoded.shape[0] == len(y_valid)
assert X_train_encoded.shape[1] == X_valid_encoded.shape[1]

print(f"Train: {X_train_encoded.shape}, {type(X_train_encoded).__name__}")
print(f"Validation: {X_valid_encoded.shape}, {type(X_valid_encoded).__name__}")

Train: (24391, 1857), csr_matrix
Validation: (24255, 1857), csr_matrix
CPU times: total: 1.25 s
Wall time: 1.45 s


In [4]:
%%time
# Задание 7. Сравнение LightGBM, CatBoost и XGBoost на one-hot признаках

from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

baseline_models = {
    "LightGBM": LGBMClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.05,
        random_state=RANDOM_STATE, n_jobs=4, verbosity=-1,
    ),
    "CatBoost": CatBoostClassifier(
        iterations=200, depth=6, learning_rate=0.05,
        random_seed=RANDOM_STATE, thread_count=4,
        verbose=False, allow_writing_files=False,
    ),
    "XGBoost": XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.05,
        tree_method="hist", random_state=RANDOM_STATE, n_jobs=4,
    ),
}

baseline_rows = []
for name, model in baseline_models.items():
    model.fit(X_train_encoded, y_train)
    scores = model.predict_proba(X_valid_encoded)[:, 1]
    auc = roc_auc_score(y_valid, scores)
    baseline_rows.append({
        "Модель": name,
        "ROC AUC": auc,
        "Gini": 2 * auc - 1,
    })

baseline_results = pd.DataFrame(baseline_rows).sort_values(
    "Gini", ascending=False
)
print(baseline_results.to_string(index=False, float_format="{:.4f}".format))

  Модель  ROC AUC   Gini
CatBoost   0.7447 0.4895
 XGBoost   0.7430 0.4860
LightGBM   0.7387 0.4773
CPU times: total: 15 s
Wall time: 6.41 s


In [5]:
%%time
# Задание 7. CatBoost со встроенной обработкой категорий

def prepare_catboost_features(X):
    categories = X[categorical_columns].fillna("__MISSING__").astype(str)
    numbers = pd.DataFrame(
        preprocessor.named_transformers_["numeric"].transform(X[numeric_columns]),
        columns=numeric_columns,
        index=X.index,
    )
    return pd.concat([categories, numbers], axis=1)


X_train_catboost = prepare_catboost_features(X_train)
X_valid_catboost = prepare_catboost_features(X_valid)

catboost_native = CatBoostClassifier(
    **baseline_models["CatBoost"].get_params()
)
catboost_native.fit(
    X_train_catboost,
    y_train,
    cat_features=categorical_columns,
)

native_scores = catboost_native.predict_proba(X_valid_catboost)[:, 1]
native_auc = roc_auc_score(y_valid, native_scores)
native_gini = 2 * native_auc - 1
one_hot_gini = baseline_results.set_index("Модель").loc["CatBoost", "Gini"]

print(f"CatBoost с one-hot: Gini = {one_hot_gini:.4f}")
print(f"CatBoost с исходными категориями: Gini = {native_gini:.4f}")
print(f"Разница: {native_gini - one_hot_gini:+.4f}")

CatBoost с one-hot: Gini = 0.4895
CatBoost с исходными категориями: Gini = 0.4888
Разница: -0.0007
CPU times: total: 40.8 s
Wall time: 14.4 s


In [6]:
%%time
# Задание 7. Быстрое сравнение XGBoost и DART на 30 деревьях

from time import perf_counter

params = baseline_models["XGBoost"].get_params()
params.update(n_estimators=30)

for booster in ("gbtree", "dart"):
    mode_params = dict(params, booster=booster)
    if booster == "dart":
        mode_params["rate_drop"] = 0.1

    print(f"Обучение {booster}...", flush=True)
    started = perf_counter()

    model = XGBClassifier(**mode_params).fit(X_train_encoded, y_train)
    scores = model.predict_proba(X_valid_encoded)[:, 1]
    gini = 2 * roc_auc_score(y_valid, scores) - 1

    print(f"{booster}: Gini = {gini:.4f}, время = {perf_counter() - started:.1f} с")

Обучение gbtree...
gbtree: Gini = 0.4753, время = 0.3 с
Обучение dart...
dart: Gini = 0.4695, время = 305.5 с
CPU times: total: 16min 17s
Wall time: 5min 5s


In [7]:
%%time
# Задание 7. Подбор глубины и числа деревьев на валидации

trials = [
    ("LightGBM", {"max_depth": 4}),
    ("LightGBM", {"n_estimators": 350}),
    ("CatBoost", {"depth": 4}),
    ("CatBoost", {"iterations": 350}),
    ("XGBoost", {"max_depth": 4}),
    ("XGBoost", {"n_estimators": 350}),
]

candidate_models = dict(baseline_models)
candidate_models["CatBoost с исходными категориями"] = catboost_native

comparison_rows = baseline_results.to_dict("records")
comparison_rows.append({
    "Модель": "CatBoost с исходными категориями",
    "ROC AUC": native_auc,
    "Gini": native_gini,
})

for name, changes in trials:
    base_model = baseline_models[name]
    params = base_model.get_params()
    params.update(changes)

    label = f"{name} ({', '.join(f'{k}={v}' for k, v in changes.items())})"
    candidate = type(base_model)(**params)
    candidate.fit(X_train_encoded, y_train)

    scores = candidate.predict_proba(X_valid_encoded)[:, 1]
    auc = roc_auc_score(y_valid, scores)

    candidate_models[label] = candidate
    comparison_rows.append({"Модель": label, "ROC AUC": auc, "Gini": 2 * auc - 1})

comparison_results = pd.DataFrame(comparison_rows).sort_values(
    "Gini", ascending=False
)
print(comparison_results.to_string(index=False, float_format="{:.4f}".format))

                          Модель  ROC AUC   Gini
       CatBoost (iterations=350)   0.7448 0.4897
              CatBoost (depth=4)   0.7448 0.4896
                        CatBoost   0.7447 0.4895
CatBoost с исходными категориями   0.7444 0.4888
                         XGBoost   0.7430 0.4860
          LightGBM (max_depth=4)   0.7429 0.4858
           XGBoost (max_depth=4)   0.7421 0.4842
      XGBoost (n_estimators=350)   0.7414 0.4828
                        LightGBM   0.7387 0.4773
     LightGBM (n_estimators=350)   0.7372 0.4744
CPU times: total: 35.4 s
Wall time: 9.5 s


In [8]:
%%time
# Задание 7. Сравнение обычных и линейных деревьев LightGBM

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler(with_mean=False)
X_train_lgbm = scaler.fit_transform(X_train_encoded)
X_valid_lgbm = scaler.transform(X_valid_encoded)

lightgbm_gini = {}
for linear_tree in (False, True):
    lightgbm_model = LGBMClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        linear_tree=linear_tree,
        random_state=RANDOM_STATE,
        n_jobs=4,
        verbosity=-1,
    ).fit(X_train_lgbm, y_train)

    scores = lightgbm_model.predict_proba(X_valid_lgbm)[:, 1]
    lightgbm_gini[linear_tree] = 2 * roc_auc_score(y_valid, scores) - 1
    print(f"linear_tree={linear_tree}: Gini = {lightgbm_gini[linear_tree]:.4f}")

delta = lightgbm_gini[True] - lightgbm_gini[False]
print(f"Разница Gini: {delta:+.4f}")

linear_tree=False: Gini = 0.4858
linear_tree=True: Gini = 0.4855
Разница Gini: -0.0004
CPU times: total: 3.34 s
Wall time: 1.11 s


In [9]:
%%time
# Задание 8. Итоговая оценка лучшей модели на train, validation и test

best_model = candidate_models["CatBoost (iterations=350)"]

df_test = pd.read_csv(DATA_DIR / "df_test.csv")
X_test = df_test.drop(columns=[TARGET])
y_test = df_test[TARGET]

assert X_test.columns.equals(X_train.columns)
X_test_encoded = preprocessor.transform(X_test)

final_rows = []
for split, X_split, y_split in (
    ("Train", X_train_encoded, y_train),
    ("Validation", X_valid_encoded, y_valid),
    ("Test", X_test_encoded, y_test),
):
    scores = best_model.predict_proba(X_split)[:, 1]
    auc = roc_auc_score(y_split, scores)
    final_rows.append({"Выборка": split, "ROC AUC": auc, "Gini": 2 * auc - 1})

final_results = pd.DataFrame(final_rows).set_index("Выборка")
print(final_results.to_string(float_format="{:.4f}".format))

gini = final_results["Gini"]
print(f"\nРазница Train − Validation: {gini['Train'] - gini['Validation']:+.4f}")
print(f"Разница Validation − Test: {gini['Validation'] - gini['Test']:+.4f}")

            ROC AUC   Gini
Выборка                   
Train        0.8149 0.6297
Validation   0.7448 0.4897
Test         0.7533 0.5067

Разница Train − Validation: +0.1400
Разница Validation − Test: -0.0170
CPU times: total: 984 ms
Wall time: 614 ms


В CatBoost параметр cat_features позволяет передать исходные категории для встроенной обработки. При одинаковых параметрах модели Gini составил 0,4888 с исходными категориями и 0,4895 с one-hot.

В LightGBM при linear_tree=True листья всех деревьев после первого используют линейные модели. При одинаковых настройках и предобработке Gini составил 0,4855 с линейными деревьями и 0,4858 с обычными. Разница по неокруглённым значениям составила −0,0004.

В XGBoost режим DART временно исключает часть предыдущих деревьев при обучении очередного. При 30 деревьях Gini составил 0,4695 у DART и 0,4753 у gbtree. В этом эксперименте DART также обучался заметно дольше.

Лучший результат на валидационной выборке показал CatBoost с one-hot и 350 деревьями. Его Gini составил 0,4897. Увеличение числа деревьев с 200 до 350 повысило Gini примерно на 0,0002.

In [10]:
%%time
# Задание 8. Сравнение признаков переобучения у трёх моделей

selected_models = [
    "CatBoost (iterations=350)",
    "XGBoost",
    "LightGBM (max_depth=4)",
]

rows = []
for name in selected_models:
    model = candidate_models[name]

    train_scores = model.predict_proba(X_train_encoded)[:, 1]
    valid_scores = model.predict_proba(X_valid_encoded)[:, 1]

    train_gini = 2 * roc_auc_score(y_train, train_scores) - 1
    valid_gini = 2 * roc_auc_score(y_valid, valid_scores) - 1

    rows.append({
        "Модель": name,
        "Gini Train": train_gini,
        "Gini Validation": valid_gini,
        "Разрыв Train − Validation": train_gini - valid_gini,
    })

overfit_results = pd.DataFrame(rows).sort_values(
    "Gini Validation", ascending=False
)
print(overfit_results.to_string(index=False, float_format="{:.4f}".format))

                   Модель  Gini Train  Gini Validation  Разрыв Train − Validation
CatBoost (iterations=350)      0.6297           0.4897                     0.1400
                  XGBoost      0.7093           0.4860                     0.2233
   LightGBM (max_depth=4)      0.6202           0.4858                     0.1344
CPU times: total: 1.84 s
Wall time: 720 ms


У всех трёх моделей Gini на обучении выше, чем на валидации. Наиболее выражен этот признак возможного переобучения у XGBoost: Gini снизился с 0,7093 до 0,4860, разрыв составил 0,2233. У CatBoost разрыв меньше, 0,1400, а у LightGBM он самый небольшой, 0,1344. Поскольку выборки разделены по времени, часть разрыва может объясняться изменением данных между периодами.

CatBoost с 350 деревьями показал лучший Gini на валидации, 0,4897. На более поздней тестовой выборке его Gini вырос до 0,5067. Падения качества относительно валидации не наблюдалось.